## Dịch gold NER VI → EN (Qwen local)

Điều khiển pipeline chuẩn trong `source/translation/` (xem `docs/pipeline.md`):
clone/cập nhật repo, mount Drive, trỏ vào checkpoint Qwen local và
`MRCONSO.RRF`, gọi `source.translation.pipeline.run_pipeline`, rồi kiểm chứng
file đã ghi.

Notebook **không chứa logic pipeline**: gắn tag, prompt, parse, validate span,
resume và ghi output đều nằm trong `source/translation/`. Notebook cũ
`source/notebooks/common/translate_gold.ipynb` chỉ còn để tham khảo.

Output nằm cạnh `OUTPUT_PATH`: `translated.jsonl` (pass), `review.jsonl`
(fail), `translated.jsonl.done.jsonl` (manifest resume), `umls_hints.pkl`
(cache hint UMLS).

In [ ]:
%pip install "transformers>=4.44" "accelerate>=0.34" "sentence-transformers>=2.7" "tqdm>=4.66"

## Import Libraries

Chỉ thư viện chuẩn ở đây; package `source.translation` được import sau khi
repo đã có trên runtime.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

print("Python:", sys.version.split()[0])

## Constants and Configuration

Sửa `DRIVE_DATA_DIR` một lần; mọi đường dẫn Drive suy ra từ đó.

`MODEL_PATH` và `UMLS_PATH` phải **đã tồn tại** — pipeline không tải model và
không gọi API ngoài. `SAP_MODEL = None` thì validation chỉ exact/canonical.

`RESUME = True` bỏ qua record đã có trong output + manifest; đặt `False` để
truncate cả ba file output và chạy lại từ đầu. `LIMIT` chỉ để smoke test.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_DATA_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "work" / "nlp" / "UMLS" / "entity_linking" / "data"

# Inputs (must already exist)
INPUT_PATH = DRIVE_DATA_DIR / "test_fixed_verified_gold_revised.jsonl"
UMLS_ROOT = DRIVE_MOUNT_POINT / "MyDrive" / "work" / "nlp" / "UMLS"
UMLS_PATH = UMLS_ROOT / "MRCONSO.RRF"    # .gz cũng được; tự tìm dưới UMLS_ROOT nếu lệch

# Translation LLM (local checkpoint, transformers only, no quantization)
MODEL_NAME = "Qwen3-27B"               # folder name under DRIVE_DATA_DIR/models
MODEL_PATH = DRIVE_DATA_DIR / "models" / MODEL_NAME
DTYPE = "bfloat16"                     # A100 native; float16 | float32 optional
MAX_SEQ_LEN = None                     # None = model max_position_embeddings
MAX_NEW_TOKENS = 2048                  # output cap; generation stops at EOS
SAP_MODEL = None        # e.g. DRIVE_DATA_DIR / "models" / "SapBERT"; None = exact only

# Outputs (OUTPUT_DIR must already exist)
OUTPUT_DIR = DRIVE_DATA_DIR
OUTPUT_PATH = OUTPUT_DIR / "translated.jsonl"
REVIEW_PATH = OUTPUT_DIR / "review.jsonl"
UMLS_CACHE = OUTPUT_DIR / "umls_hints.pkl"

# ---------------------------------------------------------------------------
# Pipeline settings (A100 80GB)
# ---------------------------------------------------------------------------
BATCH_SIZE = 32
SAP_THR = 0.85
MAX_RETRIES = 2
LIMIT = None            # None = all records; an int truncates for a smoke run
RESUME = True           # False truncates output, review and manifest

print(f"Input  : {INPUT_PATH}")
print(f"UMLS   : {UMLS_PATH} (search root: {UMLS_ROOT})")
print(f"Model  : {MODEL_NAME} -> {MODEL_PATH} ({DTYPE})")
print(f"Limits : max_seq_len={MAX_SEQ_LEN or 'model max'}, max_new_tokens={MAX_NEW_TOKENS}, batch={BATCH_SIZE}")
print(f"Output : {OUTPUT_PATH}")

## Stage 1 — Clone or Update the Repository

Package dịch nằm trong repo (`source/translation/`), không nằm trong notebook.
Chạy lại cell này sẽ cập nhật bản clone thay vì tải lại.

In [ ]:
def _git(*args: str) -> None:
    print("git", " ".join(args))
    subprocess.run(["git", *args], check=True)


if (REPO_CLONE_DIR / ".git").exists():
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_REPO_URL, str(REPO_CLONE_DIR))

if not (REPO_CLONE_DIR / "source" / "translation" / "pipeline.py").is_file():
    raise FileNotFoundError(
        f"source/translation/pipeline.py is missing from {REPO_CLONE_DIR}; "
        "check GIT_BRANCH and the repository layout."
    )
if str(REPO_CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_CLONE_DIR))

from source.translation.pipeline import PipelineConfig, run_pipeline

print("Repository ready:", REPO_CLONE_DIR)

## Stage 2 — Mount Drive and Validate the Paths

Mọi thứ phải có sẵn; notebook không tạo folder trên Drive và không tải model.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))


def require_path(path, name: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    return path


def resolve_umls(path: Path, search_root: Path) -> Path:
    if path.exists():
        return path
    matches = sorted(
        item for item in search_root.rglob("MRCONSO.RRF*") if item.is_file()
    )
    if matches:
        print(f"UMLS_PATH {path} missing; using {matches[0]}")
        return matches[0]
    raise FileNotFoundError(
        f"MRCONSO.RRF not found at {path} or anywhere under {search_root}"
    )


input_path = require_path(INPUT_PATH, "INPUT_PATH")
umls_path = resolve_umls(UMLS_PATH, UMLS_ROOT)
model_path = require_path(MODEL_PATH, "MODEL_PATH")
output_dir = require_path(OUTPUT_DIR, "OUTPUT_DIR")
if not output_dir.is_dir():
    raise NotADirectoryError(f"OUTPUT_DIR is not a directory: {output_dir}")
sap_model = str(require_path(SAP_MODEL, "SAP_MODEL")) if SAP_MODEL else None

for label, path in (("OUTPUT_PATH", OUTPUT_PATH), ("REVIEW_PATH", REVIEW_PATH), ("UMLS_CACHE", UMLS_CACHE)):
    if Path(path).parent.resolve() != output_dir.resolve():
        raise ValueError(f"{label} must live in OUTPUT_DIR, got {Path(path).parent}")

print(f"Input records : {sum(1 for line in input_path.open(encoding='utf-8-sig') if line.strip()):,}")
print(f"UMLS          : {umls_path}")
print(f"Model         : {model_path}")
print(f"SapBERT       : {sap_model or 'disabled (exact/canonical only)'}")

## Stage 3 — Run the Pipeline

`run_pipeline` đọc record, gắn tag, dịch theo batch, retry khi validation fail,
và ghi `translated.jsonl` / `review.jsonl` / manifest ngay khi từng batch xong.
Ngắt runtime rồi chạy lại với `RESUME = True` sẽ tiếp tục, không dịch lại.

In [ ]:
config = PipelineConfig(
    model=str(model_path),
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=MAX_SEQ_LEN,
    dtype=DTYPE,
    umls_path=str(umls_path),
    sap_thr=SAP_THR,
    input_path=str(input_path),
    output_path=str(OUTPUT_PATH),
    review_path=str(REVIEW_PATH),
    umls_cache=str(UMLS_CACHE),
    sap_model=sap_model,
    max_retries=MAX_RETRIES,
    resume=RESUME,
    limit=LIMIT,
)

started = time.time()
stats = run_pipeline(config)
print(json.dumps(stats.as_dict(), ensure_ascii=False, indent=2))
print(f"Wall clock: {time.time() - started:.1f}s")

## Stage 4 — Verify the Written Files

Kiểm những gì consumer dựa vào: mỗi dòng output có `record_index`,
`en_sentence_str`, và token span EN nằm trong câu theo đúng tokenizer của
pipeline (`parse.tokenize_en`, không phải `str.split()`).

Record fail nằm trong `review.jsonl`; chạy lại với `RESUME = True` sẽ thử lại
chúng, vì review không được dùng để skip.

In [ ]:
from source.translation.parse import tokenize_en


def read_jsonl(path) -> list[dict]:
    path = Path(path)
    if not path.is_file():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


translated = read_jsonl(OUTPUT_PATH)
review = read_jsonl(REVIEW_PATH)

for row in translated:
    record_index = row["record_index"]
    tokens = tokenize_en(row["en_sentence_str"])
    for entity in row["entities"]:
        assert entity.get("en_term"), (record_index, "missing en_term")
        start = entity.get("en_start_token_idx")
        end = entity.get("en_end_token_idx")
        if start is None or end is None:
            continue
        assert 0 <= start <= end < len(tokens), (record_index, start, end, len(tokens))

print(f"translated: {len(translated):,} rows")
print(f"review    : {len(review):,} rows")
if review:
    reasons: dict[str, int] = {}
    for row in review:
        reasons[row["reason"]] = reasons.get(row["reason"], 0) + 1
    print("review reasons:", json.dumps(reasons, ensure_ascii=False, indent=2))
if translated:
    print("first row:", json.dumps(translated[0], ensure_ascii=False)[:600])